# Arabic Lecture Speech-to-Text → Translation Pipeline

### Pipeline Overview
```
Arabic Audio Upload
      ↓
Audio Preprocessing (optional noise reduction + 16kHz normalization)
      ↓
Whisper large-v3-turbo (Arabic transcription, anti-hallucination tuned)
      ↓
Arabic Text Normalization (unify Hamza, Tashkeel, Alef forms)
      ↓
PDF Extraction (text + OCR fallback for image-based slides)
      ↓
Gemini: Extract lecture-specific glossary from PDF
      ↓
Gemini: Correct transcript using PDF + glossary
  (falls back to OpenRouter → then original text)
      ↓
Gemini: Translate Arabic → English
  (same fallback chain)
      ↓
Save: Arabic transcript + English translation + glossary + full JSON

```

## Install Dependencies

In [1]:
import gc
import torch

for var in [
    "model",
    "pipe",
    "trainer",
    "tokenizer",
    "processor",
    "inputs",
    "outputs",
]:
    if var in globals():
        del globals()[var]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("GPU memory released.")

GPU memory released.


In [2]:
!pip install -q openai-whisper
!pip install -q pydub
!pip install -q librosa
!pip install -q soundfile
!pip install -q pyarabic

!apt-get install -y -q ffmpeg

print("\n All dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.4 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 80.8 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cud

## Upload Arabic Audio File

Supported formats: `mp3`, `wav`, `m4a`, `ogg`, `flac`, `mp4`

In [3]:
import os
import warnings
warnings.filterwarnings("ignore")

KAGGLE_AUDIO_PATH = "/kaggle/input/datasets/farahlatrache/datasetstt/L03_ CH8 Virtual Memory Concepts  and Multilevel Paging.mp3"
KAGGLE_PDF_PATH   = "/kaggle/input/datasets/farahlatrache/datasetstt/03 Virtual Memory I.pdf"   # optional

IS_KAGGLE = os.path.exists("/kaggle")
AUDIO_PATH = KAGGLE_AUDIO_PATH if IS_KAGGLE else "/tmp/lecture.mp3"
PDF_PATH   = KAGGLE_PDF_PATH   if IS_KAGGLE else ""

print(f" Audio : {AUDIO_PATH}")
print(f" PDF   : {PDF_PATH if PDF_PATH else '(none — PDF correction will be skipped)'}")
print(f" Env   : {'Kaggle' if IS_KAGGLE else 'Local'}")


 Audio : /kaggle/input/datasets/farahlatrache/datasetstt/L03_ CH8 Virtual Memory Concepts  and Multilevel Paging.mp3
 PDF   : /kaggle/input/datasets/farahlatrache/datasetstt/03 Virtual Memory I.pdf
 Env   : Kaggle


## Audio Preprocessing

In [4]:
import numpy as np
import librosa
import soundfile as sf
from pydub import AudioSegment

SAMPLE_RATE     = 16000
PROCESSED_PATH  = "/tmp/arabic_processed.wav"
NOISE_REDUCTION = False

def preprocess_audio(input_path: str, output_path: str, apply_nr: bool = False):
    """
    Preprocess audio for Whisper:
      1. Convert any format → WAV via pydub
      2. Resample to 16kHz mono via librosa
      3. Spectral noise reduction (optional, off by default)
      4. Normalize amplitude
    """
    print(f" Loading: {input_path}")

    ext = os.path.splitext(input_path)[1].lower().lstrip(".")
    if ext != "wav":
        print(f"   Converting {ext.upper()} → WAV ...")
        tmp_wav = "/tmp/arabic_tmp_input.wav"
        AudioSegment.from_file(input_path).export(tmp_wav, format="wav")
        load_path = tmp_wav
    else:
        load_path = input_path

    print(f"   Resampling to {SAMPLE_RATE} Hz mono ...")
    audio, sr = librosa.load(load_path, sr=SAMPLE_RATE, mono=True)
    duration  = len(audio) / sr
    print(f"   Duration: {duration:.1f}s ({duration/60:.1f} min)")

    if apply_nr and duration > 0.5:
        try:
            import noisereduce as nr
            print("   Applying noise reduction ...")
            noise_sample = audio[: int(min(0.5, duration * 0.08) * sr)]
            audio = nr.reduce_noise(
                y=audio, sr=sr, y_noise=noise_sample,
                stationary=False, prop_decrease=0.7
            )
        except ImportError:
            print("      noisereduce not installed — skipping. Run: pip install noisereduce")

    print("   Normalizing amplitude ...")
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val * 0.95

    sf.write(output_path, audio, sr)
    print(f" Preprocessed audio saved → {output_path}")
    return output_path, duration


processed_audio, duration = preprocess_audio(AUDIO_PATH, PROCESSED_PATH, NOISE_REDUCTION)


 Loading: /kaggle/input/datasets/farahlatrache/datasetstt/L03_ CH8 Virtual Memory Concepts  and Multilevel Paging.mp3
   Converting MP3 → WAV ...
   Resampling to 16000 Hz mono ...
   Duration: 3185.6s (53.1 min)
   Normalizing amplitude ...
 Preprocessed audio saved → /tmp/arabic_processed.wav


## Load Whisper large-v3-turbo

In [7]:
import whisper
import torch 
WHISPER_MODEL_SIZE = "large-v3-turbo"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device: {device.upper()}")

if device == "cpu":
    print("     No GPU detected! large-v3-turbo on CPU will still be slow.")
    print("   → On Kaggle: Settings → Accelerator → GPU T4 x2")

model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)
print(f" Whisper {WHISPER_MODEL_SIZE} loaded on {device.upper()}")


  Device: CUDA
 Whisper large-v3-turbo loaded on CUDA


## Transcribe Arabic Audio

In [8]:
ARABIC_INITIAL_PROMPT = """
هذه محاضرة جامعية في علوم الحاسب. قد تحتوي على مصطلحات تقنية باللغة الإنجليزية مثل:

Algorithms, Data Structures, Complexity, Big O, Recursion, Dynamic Programming,
Graph Theory, Dijkstra, BFS, DFS, Minimum Spanning Tree, Kruskal, Prim,
Convex Hull, Triangulation, Voronoi, Delaunay, Sweep Line, Segment Intersection,
Computational Geometry, Polygon, Vertex, Edge,

Machine Learning, Deep Learning, Neural Network, Transformer, Attention,
Backpropagation, Gradient Descent, Convolution, Pooling, Feature Map,
Computer Vision, Object Detection, Segmentation, Laplacian, Gaussian, Sobel,
Harris, Corner Detection, Descriptor, Keypoint, SIFT, HOG,
Natural Language Processing, Tokenization, Embedding, BERT, GPT,
POS Tagging, Named Entity Recognition, Sentiment Analysis,

Human Computer Interaction, Biometrics, ECG, EEG, ECoG, fMRI,
FAR, FRR, EER, BCI, P300, SSVEP,

Cryptography, Encryption, Decryption, RSA, AES, DES, Cipher,
Public Key, Private Key, Hash, Digital Signature, Elliptic Curve,
Extended Euclidean, Fermat, Euler, Modular Arithmetic, Blum Blum Shub,
Hill Cipher, Playfair, Autokey, CBC, ECB, IV,

Robotics, Sensor, Actuator, PID, Path Planning, Potential Field,
Cell Decomposition, Swarm, Foraging, Kalman Filter,

Operating Systems, Process, Thread, Scheduler, Deadlock, Semaphore,
Virtual Memory, Paging, File System,

Software Engineering, Requirements, Use Case, SRS, UML, Agile, Sprint,
Database, SQL, NoSQL, REST, API, HTTP, TCP, IP,

Network, Firewall, Protocol, Packet, Router, Encryption.

اكتب المصطلحات التقنية الإنجليزية كما تُنطق ولا تحاول ترجمتها للعربية.
"""

# ── Technical terms dictionary ───────────────────────────────────────────────

TECHNICAL_TERMS = [
    # Core CS / Algorithms
    "Algorithm", "Complexity", "Big O", "Recursion", "Dynamic Programming",
    "Greedy", "Backtracking", "Divide and Conquer", "Memoization",
    # Data Structures
    "Stack", "Queue", "Heap", "Tree", "Binary Search Tree", "AVL Tree",
    "Hash Table", "Graph", "Adjacency Matrix", "Adjacency List", "Trie",
    # Graph Algorithms
    "Dijkstra", "Bellman-Ford", "Floyd-Warshall", "BFS", "DFS", "Topological Sort",
    "Minimum Spanning Tree", "Kruskal", "Prim",
    # Computational Geometry
    "Convex Hull", "Triangulation", "Voronoi Diagram", "Delaunay Triangulation",
    "Sweep Line", "Segment Intersection", "Polygon", "Vertex", "Half-Edge",
    "Point Location", "Fortune Algorithm",
    # Computer Vision / Image Processing
    "Gradient", "Sobel", "Laplacian", "Gaussian", "Convolution", "Kernel",
    "Feature", "Descriptor", "Harris", "Corner Detection", "SIFT", "HOG",
    "Edge Detection", "Thresholding", "Morphological", "Histogram",
    # Machine Learning / Deep Learning
    "Neural Network", "Transformer", "Attention", "Backpropagation",
    "Gradient Descent", "Optimizer", "Loss Function", "Overfitting",
    "Regularization", "Dropout", "Batch Normalization", "Convolutional",
    "Pooling", "Feature Map", "Embedding", "Fine-tuning", "Transfer Learning",
    # NLP
    "Tokenization", "POS Tagging", "Named Entity Recognition",
    "Sentiment Analysis", "BERT", "GPT", "Word2Vec", "TF-IDF",
    # HCI / Biometrics
    "FAR", "FRR", "EER", "ECG", "EEG", "ECoG", "fMRI", "BCI", "P300", "SSVEP",
    "Biometric", "Authentication",
    # Cryptography
    "Encryption", "Decryption", "Cipher", "RSA", "AES", "DES", "Hash",
    "Digital Signature", "Public Key", "Private Key", "Elliptic Curve",
    "Extended Euclidean", "Modular Arithmetic", "Blum Blum Shub",
    "Hill Cipher", "Playfair", "Autokey", "CBC", "ECB",
    # Robotics
    "Sensor", "Actuator", "PID", "Path Planning", "Potential Field",
    "Cell Decomposition", "Swarm", "Kalman Filter", "Localization", "SLAM",
    # Software Engineering
    "Requirements", "Use Case", "SRS", "UML", "Agile", "Sprint", "Scrum",
    "REST", "API", "Microservice",
    # OS / Systems
    "Process", "Thread", "Scheduler", "Deadlock", "Semaphore",
    "Virtual Memory", "Paging", "Cache", "Pipeline",
]

terms_hint = ", ".join(TECHNICAL_TERMS[:40])

ARABIC_INITIAL_PROMPT_FULL = ARABIC_INITIAL_PROMPT + f"\n\nبعض المصطلحات المتوقعة: {terms_hint}."

import re as _re_hallucination_check

def has_repetition_hallucination(text: str, min_repeats: int = 4) -> bool:
    """
    Detects Whisper's classic failure mode: a single word/short phrase
    repeated many times in a row (e.g. "\u0627\u0646\u0627 \u0627\u0646\u0627 \u0627\u0646\u0627 \u0627\u0646\u0627").
    Usually triggered by silence, music, or low-quality audio segments.
    """
    words = text.split()
    if len(words) < min_repeats:
        return False
    for i in range(len(words) - min_repeats + 1):
        window = words[i:i + min_repeats]
        if len(set(window)) == 1:
            return True
    return False


def collapse_repetitions(text: str, max_repeats: int = 2) -> str:
    """Collapse any run of an identical word repeated >max_repeats times down to max_repeats."""
    words = text.split()
    out = []
    i = 0
    while i < len(words):
        j = i
        while j < len(words) and words[j] == words[i]:
            j += 1
        run_len = j - i
        if run_len > max_repeats:
            out.extend([words[i]] * max_repeats)
        else:
            out.extend(words[i:j])
        i = j
    return " ".join(out)


def collapse_repeated_segments(segments: list, max_repeats: int = 2) -> tuple:
    """
    Detects Whisper's OTHER hallucination shape: the same phrase repeated
    across multiple consecutive SEGMENTS (not just within one segment's text).
    e.g. 5 segments in a row all saying "\u0627\u0644 special \u0627\u0644\u064a \u0647\u064a" \u2014 this happens
    when Whisper gets stuck on a bad audio patch and keeps re-emitting the
    same short phrase as a "new" segment instead of looping within one.

    Keeps at most max_repeats consecutive duplicate segments and merges their
    timestamps into a single span; drops the rest. Returns (cleaned_segments, count_dropped).
    """
    if not segments:
        return segments, 0

    def normalize(t: str) -> str:
        return " ".join(t.strip().split())

    cleaned = []
    dropped = 0
    i = 0
    while i < len(segments):
        run = [segments[i]]
        j = i + 1
        while j < len(segments) and normalize(segments[j]["text"]) == normalize(segments[i]["text"]):
            run.append(segments[j])
            j += 1

        if len(run) > max_repeats:
            kept = run[:max_repeats]
            kept[-1] = dict(kept[-1])
            kept[-1]["end"] = run[-1]["end"]
            cleaned.extend(kept)
            dropped += len(run) - max_repeats
        else:
            cleaned.extend(run)

        i = j

    return cleaned, dropped

TEMPERATURE_FALLBACK = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)

result = model.transcribe(
    processed_audio,
    language                    = "ar",
    task                        = "transcribe",
    initial_prompt              = ARABIC_INITIAL_PROMPT_FULL,
    condition_on_previous_text  = False,   # prevents repetition-loop lock-in across segments
    word_timestamps             = False,
    temperature                 = TEMPERATURE_FALLBACK,  # retries with more randomness on failure
    compression_ratio_threshold = 2.4,     # stricter — catches repetitive text earlier (was 2.6)
    logprob_threshold           = -1.0,    # reject low-confidence segments -> triggers fallback
    no_speech_threshold         = 0.6,     # slightly stricter silence detection
    verbose                     = True,
)

segments      = result["segments"]
raw_text      = result["text"]
detected_lang = result.get("language", "ar")

hallucinated_segments = 0
for seg in segments:
    if has_repetition_hallucination(seg["text"]):
        hallucinated_segments += 1
        seg["text"] = collapse_repetitions(seg["text"])

segments, dropped_duplicate_segments = collapse_repeated_segments(segments)
if dropped_duplicate_segments:
    print(f"\n  Collapsed {dropped_duplicate_segments} duplicate consecutive segment(s) "
          f"(cross-segment repetition loop).")

if hallucinated_segments or dropped_duplicate_segments:
    raw_text = " ".join(seg["text"] for seg in segments)
if hallucinated_segments:
    print(f"  Detected and cleaned {hallucinated_segments} within-segment repetition hallucination(s).")

print(f"\n Transcription complete!")
print(f"   Language detected           : {detected_lang}")
print(f"   Total segments (final)      : {len(segments)}")
print(f"   Raw text length             : {len(raw_text)} chars")
print(f"   Within-segment repetitions  : {hallucinated_segments} (auto-cleaned)")
print(f"   Duplicate segments dropped  : {dropped_duplicate_segments} (cross-segment loop, auto-cleaned)")


[00:00.000 --> 00:06.000]  السلام عليكم ورحمة الله. كلها عملت بخير جميعاً. رمضان كريم علينا وعليكم إن شاء الله.
[00:06.000 --> 00:23.000]  استكمال جزئية الميموري النهاردة إن شاء الله بس بنتكلم على الـ Virtual Memory. اللي هو إزاي الميموري تتأسم ما بين الرمات للـ Physical Memory والـ Extinction بتاعها اللي هو الـ Hard Disk.
[00:23.000 --> 00:26.000]  ناخد جزء من الهارد ونعمله كأين هو ميموري
[00:26.000 --> 00:30.000]  بنتعرف النهاردة على بعض الكونسبت اللي مخليها
[00:30.000 --> 00:33.000]  دي فيزيبول يعني وايمة لحد دلوقتي
[00:33.000 --> 00:36.000]  وحنشوف بداي
[00:36.000 --> 00:39.000]  احنا لنى عملية التحويل بتبقى ما بين
[00:39.000 --> 00:41.000]  في كل اتنين
[00:41.000 --> 00:44.000]  الدور على التحويل ثم بين اتنين
[00:44.000 --> 00:47.000]  الهاردوير اللي هو ممثل في الميموري منيج منت
[00:47.000 --> 00:49.000]  وده اللي بنتعرف عنه النهاردة ان شاء الله
[00:49.000 --> 00:52.000]  والأبرينيك سيستم ده اللي بنتشوفه المرة الجاية
[00:52.000 --> 00:56.240]  المرة الجاية. بازن الله.
[01:01.240 -

In [9]:
detected_dialect_code = "UNK"
detected_dialect_name = "Unknown / Mixed"
dialect_confidence    = 0.0
print("  Dialect detection skipped.")


  Dialect detection skipped.


## Arabic Text Normalization


| Issue | Example | Fix |
|-------|---------|-----|
| Tashkeel (diacritics) | كَتَبَ | Remove → كتب |
| Alef forms | إ / أ / آ / ا | Normalize → ا |
| Taa Marbouta | ة vs ه | Normalize → ة |
| Tatweel (kashida) | كـتـب | Remove → كتب |
| Hamza variants | ئ / ؤ / ء | Keep meaningful, strip noise |
| Waw with Hamza | ؤ | Normalize |
| Mixed numerals | ١٢٣ vs 123 | Unify to Arabic-Indic or Western |
| Extra spaces | Multiple spaces | Collapse |

In [10]:
import re
import unicodedata

try:
    import pyarabic.araby as araby
    PYARABIC_AVAILABLE = True
except ImportError:
    PYARABIC_AVAILABLE = False
    print("   pyarabic not available — using fallback regex normalization")


def normalize_arabic_text(text: str) -> str:
    """
    Normalize Arabic text output from Whisper.

    Operations:
      1. Remove tashkeel (diacritics/harakat) — Whisper sometimes adds them
      2. Normalize Alef variants (أ إ آ → ا)
      3. Remove tatweel (elongation character ـ)
      4. Normalize Hamza variants
      5. Normalize Arabic-Indic numerals → Western (0-9)
      6. Collapse extra whitespace / blank lines
    """
    if PYARABIC_AVAILABLE:
        text = araby.strip_tashkeel(text)
        text = araby.normalize_alef(text)
        text = araby.strip_tatweel(text)
        text = araby.normalize_hamza(text)
    else:
        text = re.sub(r'[\u064B-\u065F]', '', text)   
        text = re.sub(r'[أإآ]', 'ا', text)              
        text = re.sub(r'ـ', '', text)                   
        text = re.sub(r'ؤ', 'و', text)                

    # Arabic-Indic → Western numerals
    text = text.translate(str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789'))

    # Collapse whitespace
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


print("   Normalizing Arabic text ...")
normalized_raw = normalize_arabic_text(raw_text)
print(f"  Normalization done.")
print(f"   Before: {len(raw_text)} chars | After: {len(normalized_raw)} chars")


   Normalizing Arabic text ...
  Normalization done.
   Before: 29813 chars | After: 29049 chars


## Segment Post-Processing


In [11]:
def format_timestamp(seconds: float) -> str:
    hours   = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs    = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def is_arabic_text(text: str) -> bool:
    """Check if a segment actually contains Arabic script (not just noise/punctuation)."""
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    return arabic_chars >= 2  # at least 2 Arabic characters


def postprocess_arabic_segments(segments: list, silence_gap: float = 2.5) -> dict:
    """
    Process Whisper segments into structured Arabic transcript.
    
    Arabic-specific adjustments:
      - RTL text doesn't affect processing, but we validate Arabic script presence
      - Slightly larger silence_gap (2.5s vs 2.0s for English) since Arabic speech
        naturally has longer pauses between phrases
      - Apply normalization per segment before merging
    """
    timestamped = []
    paragraphs  = []
    current_paragraph = []
    prev_end = 0.0

    for seg in segments:
        start = seg["start"]
        end   = seg["end"]
        text  = seg["text"].strip()

        if not text or not is_arabic_text(text):
            continue

        normalized_seg = normalize_arabic_text(text)
        if not normalized_seg:
            continue

        timestamped.append({
            "start":     format_timestamp(start),
            "end":       format_timestamp(end),
            "start_sec": round(start, 2),
            "end_sec":   round(end, 2),
            "text":      normalized_seg
        })

        gap = start - prev_end
        if gap >= silence_gap and current_paragraph:
            paragraphs.append(" ".join(current_paragraph))
            current_paragraph = []

        current_paragraph.append(normalized_seg)
        prev_end = end

    if current_paragraph:
        paragraphs.append(" ".join(current_paragraph))

    merged = "\n\n".join(paragraphs)

    final  = normalize_arabic_text(merged)

    return {
        "timestamped_transcript": timestamped,
        "paragraph_text":         merged,
        "clean_text":             final,
        "total_segments":         len(timestamped),
        "total_paragraphs":       len(paragraphs)
    }


output = postprocess_arabic_segments(segments, silence_gap=2.5)

print(" Post-processing complete!")
print(f"   Segments   : {output['total_segments']}")
print(f"   Paragraphs : {output['total_paragraphs']}")

 Post-processing complete!
   Segments   : 723
   Paragraphs : 16


## View Results

In [12]:
print("=" * 65)
print(" TIMESTAMPED ARABIC TRANSCRIPT")
print("=" * 65)
for seg in output["timestamped_transcript"]:
    print(f"[{seg['start']} → {seg['end']}]  {seg['text']}")

print()
print("=" * 65)
print(" CLEAN NORMALIZED TEXT (ready for summary model)")
print("=" * 65)
print(output["clean_text"])

print()
print("=" * 65)
print(" DIALECT INFO")
print("=" * 65)
print(f"  Detected dialect : {detected_dialect_name} [{detected_dialect_code}]")
print(f"  Confidence       : {dialect_confidence}%")

 TIMESTAMPED ARABIC TRANSCRIPT
[00:00:00 → 00:00:06]  السلام عليكم ورحمة الله. كلها عملت بخير جميعا. رمضان كريم علينا وعليكم ان شاء الله.
[00:00:06 → 00:00:23]  استكمال جزءية الميموري النهاردة ان شاء الله بس بنتكلم علا ال Virtual Memory. اللي هو ازاي الميموري تتاسم ما بين الرمات لل Physical Memory وال Extinction بتاعها اللي هو ال Hard Disk.
[00:00:23 → 00:00:26]  ناخد جزء من الهارد ونعمله كاين هو ميموري
[00:00:26 → 00:00:30]  بنتعرف النهاردة علا بعض الكونسبت اللي مخليها
[00:00:30 → 00:00:33]  دي فيزيبول يعني وايمة لحد دلوقتي
[00:00:33 → 00:00:36]  وحنشوف بداي
[00:00:36 → 00:00:39]  احنا لنا عملية التحويل بتبقا ما بين
[00:00:39 → 00:00:41]  في كل اتنين
[00:00:41 → 00:00:44]  الدور علا التحويل ثم بين اتنين
[00:00:44 → 00:00:47]  الهاردوير اللي هو ممثل في الميموري منيج منت
[00:00:47 → 00:00:49]  وده اللي بنتعرف عنه النهاردة ان شاء الله
[00:00:49 → 00:00:52]  والابرينيك سيستم ده اللي بنتشوفه المرة الجاية
[00:00:52 → 00:00:56]  المرة الجاية. بازن الله.
[00:01:01 → 00:01:04]  فاحنا كنا خلصنا

In [13]:
#Translation + PDF dependencies
!pip install -q pymupdf pillow pytesseract

import subprocess
subprocess.run(["apt-get", "install", "-qq", "-y",
                "tesseract-ocr", "tesseract-ocr-ara", "tesseract-ocr-eng"],
               check=True)

print(" PDF + OCR dependencies ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 67.0 MB/s eta 0:00:00:00:0100:01
Selecting previously unselected package tesseract-ocr-ara.
(Reading database ... 125128 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-ara_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-ara (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-ara (1:4.00~git30-7274cfa-1.1) ...
 PDF + OCR dependencies ready.


### Configure translation mode


In [ ]:
import os
import textwrap

#CONFIG 
TRANSLATION_MODE    = "gemini"   # "gemini" or "local"

GEMINI_API_KEYS = [
    "",
    "",
]

GEMINI_MODEL         = "gemini-2.5-flash-lite"

OPENROUTER_API_KEY   = ""

OPENROUTER_MODEL = "qwen/qwen-3-coder:free"

CHUNK_SIZE_WORDS    = 800
USE_PDF_CORRECTION  = True
OCR_DPI             = 200
MIN_TEXT_CHARS      = 30


def extract_pdf_text(pdf_path: str, max_chars: int = 8000) -> str:
    """
    Extract text from a lecture PDF with OCR fallback.

    Strategy per page:
      1. Try PyMuPDF direct text extraction (fast).
      2. If page returns < MIN_TEXT_CHARS → image-based slide
         → rasterize at OCR_DPI → run Tesseract OCR (ara+eng).
    Returns combined text, truncated to max_chars.
    """
    if not pdf_path or not os.path.exists(pdf_path):
        print("     No PDF path provided — skipping PDF extraction.")
        return ""

    try:
        import fitz
        import pytesseract
        from PIL import Image
        import io

        doc        = fitz.open(pdf_path)
        all_text   = []
        text_pages = 0
        ocr_pages  = 0

        print(f"     Processing {len(doc)} PDF pages ...")

        for page_num, page in enumerate(doc, 1):
            page_text = page.get_text().strip()

            if len(page_text) >= MIN_TEXT_CHARS:
                all_text.append(page_text)
                text_pages += 1

            else:
                # Image-based page → rasterize → OCR
                mat      = fitz.Matrix(OCR_DPI / 72, OCR_DPI / 72)   # scale to target DPI
                pix      = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
                img_data = pix.tobytes("png")
                img      = Image.open(io.BytesIO(img_data))

                # Tesseract with Arabic + English — handles mixed-script slides
                ocr_text = pytesseract.image_to_string(
                    img,
                    lang   = "ara+eng",
                    config = "--psm 3"   # fully automatic page segmentation
                ).strip()

                if ocr_text:
                    all_text.append(ocr_text)
                ocr_pages += 1

                if page_num % 5 == 0:
                    print(f"      ... OCR page {page_num}/{len(doc)}")

        doc.close()
        combined = "\n\n".join(all_text)
        print(f"    PDF done: {text_pages} text pages, {ocr_pages} OCR pages, "
              f"{len(combined)} chars total")

        # Sample start + end so long slide decks aren't truncated to only first few slides
        if len(combined) <= max_chars:
            return combined
        half = max_chars // 2
        sampled = combined[:half] + "\n\n[...middle omitted...]\n\n" + combined[-half:]
        print(f"     PDF sampled: first+last {half} chars (total was {len(combined)} chars)")
        return sampled

    except Exception as e:
        print(f"     PDF extraction failed: {e}")
        return ""


PDF_TEXT = extract_pdf_text(PDF_PATH) if USE_PDF_CORRECTION else ""
if PDF_TEXT:
    print(f"\n PDF reference ready ({len(PDF_TEXT)} chars) — will guide transcript correction.")
else:
    print("\n  No PDF text — correction step will be skipped.")

# TERMS_STR used by build_translation_prompt() in the next cell
TERMS_STR = ", ".join(TECHNICAL_TERMS)

     Processing 91 PDF pages ...
      ... OCR page 30/91
      ... OCR page 60/91
    PDF done: 86 text pages, 5 OCR pages, 21608 chars total
     PDF sampled: first+last 4000 chars (total was 21608 chars)

 PDF reference ready (8026 chars) — will guide transcript correction.


In [15]:
import requests
import time


def chunk_text(text: str, max_words: int = 800) -> list[str]:
    """Split text into chunks at paragraph boundaries, hard-splitting any
    paragraph that's larger than max_words on its own."""
    paragraphs = text.split("\n\n")
    chunks, cur, cur_wc = [], [], 0

    def flush():
        nonlocal cur, cur_wc
        if cur:
            chunks.append("\n\n".join(cur))
            cur, cur_wc = [], 0

    for para in paragraphs:
        wc = len(para.split())
        if wc > max_words:
            flush()
            words = para.split()
            for i in range(0, len(words), max_words):
                chunks.append(" ".join(words[i:i + max_words]))
            continue
        if cur_wc + wc > max_words and cur:
            flush()
        cur.append(para)
        cur_wc += wc
    flush()
    return chunks


api_call_stats = {
    "total_calls":       0,
    "successful_calls":  0,
    "retried_calls":     0,
    "failed_chunks":     0,
    "openrouter_calls":  0,
}


MIN_SECONDS_BETWEEN_CALLS = 13
_last_call_time = [0.0]

def _wait_for_rate_limit():
    elapsed = time.time() - _last_call_time[0]
    remaining = MIN_SECONDS_BETWEEN_CALLS - elapsed
    if remaining > 0:
        time.sleep(remaining)
    _last_call_time[0] = time.time()


_key_state = {"idx": 0, "keys": [k for k in GEMINI_API_KEYS if k]}

def _current_gemini_key() -> str:
    if not _key_state["keys"]:
        raise RuntimeError("No Gemini API keys configured in GEMINI_API_KEYS.")
    return _key_state["keys"][_key_state["idx"]]

def _rotate_gemini_key() -> bool:
    """Switch to the next key. Returns False if no more keys are left."""
    if _key_state["idx"] < len(_key_state["keys"]) - 1:
        _key_state["idx"] += 1
        print(f"    Key {_key_state['idx']} exhausted — switching to backup key "
              f"#{_key_state['idx'] + 1}/{len(_key_state['keys'])}")
        return True
    return False


MIN_SECONDS_BETWEEN_OPENROUTER_CALLS = 5
_last_openrouter_call_time = [0.0]

def _wait_for_openrouter_rate_limit():
    elapsed = time.time() - _last_openrouter_call_time[0]
    remaining = MIN_SECONDS_BETWEEN_OPENROUTER_CALLS - elapsed
    if remaining > 0:
        time.sleep(remaining)
    _last_openrouter_call_time[0] = time.time()


def openrouter_request(system_prompt: str, user_text: str, max_retries: int = 3) -> str | None:
    """
    Fallback call to OpenRouter's free tier. Returns None on failure (caller
    decides what to do) instead of raising, since this is itself a fallback
    path -- it should never be the thing that crashes the run.
    Requires OPENROUTER_API_KEY to be set in the config cell.

    Prints the actual reason for failure (model not found, auth error, rate
    limit, etc.) instead of silently swallowing it -- free model IDs on
    OpenRouter change/rotate, so visibility here matters.
    """
    if not OPENROUTER_API_KEY:
        return None

    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_text},
        ],
    }

    for attempt in range(1, max_retries + 1):
        _wait_for_openrouter_rate_limit()
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
            print(f"        OpenRouter connection error: {e}")
            time.sleep(min(2 ** attempt * 3, 30))
            continue

        if resp.status_code == 404:
            # Model ID doesn't exist / isn't available -- retrying won't help.
            print(f"        OpenRouter 404: model '{OPENROUTER_MODEL}' not found. "
                  f"Check https://openrouter.ai/models?fmt=cards&supported_parameters=tools&order=newest "
                  f"with the Free filter and update OPENROUTER_MODEL.")
            return None
        if resp.status_code == 401:
            print(f"        OpenRouter 401: invalid OPENROUTER_API_KEY.")
            return None
        if resp.status_code in (429, 503):
            print(f"        OpenRouter HTTP {resp.status_code} — retrying ({attempt}/{max_retries}) ...")
            time.sleep(min(2 ** attempt * 5, 60))
            continue

        try:
            resp.raise_for_status()
            data = resp.json()
            return data["choices"][0]["message"]["content"].strip()
        except Exception as e:
            print(f"        OpenRouter unexpected response: {e} | body: {resp.text[:200]}")
            time.sleep(min(2 ** attempt * 3, 30))
            continue

    return None


def gemini_request(system_prompt: str, user_text: str,
                    max_retries: int = 5, allow_fallback: str = None) -> str:
    """
    Gemini REST call with key rotation + global rate limiting + retry/backoff.

    On a 429, rotates to the next configured Gemini key (if any) and retries
    immediately, instead of waiting through a cooldown on an exhausted key.

    On exhausting all Gemini keys/retries:
      1. Try OpenRouter (free model) if OPENROUTER_API_KEY is set -- a
         different provider with an independent quota pool.
      2. If that also fails (or isn't configured) and allow_fallback was
         given, return allow_fallback instead of raising, so one stubborn
         chunk doesn't crash a multi-hour pipeline run.
    """
    for attempt in range(1, max_retries + 1):
        _wait_for_rate_limit()
        api_key = _current_gemini_key()
        url = (
            "https://generativelanguage.googleapis.com/v1beta"
            f"/models/{GEMINI_MODEL}:generateContent?key={api_key}"
        )
        payload = {
            "system_instruction": {"parts": [{"text": system_prompt}]},
            "contents":           [{"parts": [{"text": user_text}]}],
            "generationConfig": {
                "maxOutputTokens": 8192,
                "thinkingConfig": {"thinkingBudget": 0},
            },
        }

        api_call_stats["total_calls"] += 1
        if attempt > 1:
            api_call_stats["retried_calls"] += 1
        try:
            resp = requests.post(url, json=payload, timeout=120)
        except requests.exceptions.ReadTimeout:
            wait = min(2 ** attempt * 5, 90)
            print(f"     Read timeout — retrying in {wait}s (attempt {attempt}/{max_retries}) ...")
            time.sleep(wait)
            continue
        except requests.exceptions.ConnectionError:
            wait = min(2 ** attempt * 5, 90)
            print(f"     Connection error — retrying in {wait}s (attempt {attempt}/{max_retries}) ...")
            time.sleep(wait)
            continue

        if resp.status_code == 429:
            if _rotate_gemini_key():
                continue   # retry immediately with the new key, no cooldown needed
            wait = 65
            print(f"     HTTP 429 (rate limit, no backup keys left) — cooling down {wait}s "
                  f"(attempt {attempt}/{max_retries}) ...")
            time.sleep(wait)
            continue
        if resp.status_code == 503:
            wait = min(2 ** attempt * 5, 90)
            print(f"     HTTP 503 — retrying in {wait}s (attempt {attempt}/{max_retries}) ...")
            time.sleep(wait)
            continue

        resp.raise_for_status()
        data = resp.json()
        try:
            result_text = data["candidates"][0]["content"]["parts"][0]["text"].strip()
            api_call_stats["successful_calls"] += 1
            return result_text
        except Exception:
            wait = min(2 ** attempt * 5, 90)
            print(f"     Empty/unexpected response — retrying in {wait}s (attempt {attempt}/{max_retries}) ...")
            time.sleep(wait)
            continue

    if OPENROUTER_API_KEY:
        print(f"    Gemini exhausted — trying OpenRouter ({OPENROUTER_MODEL}) ...")
        or_result = openrouter_request(system_prompt, user_text)
        if or_result:
            print(f"    OpenRouter succeeded.")
            api_call_stats["openrouter_calls"] += 1
            return or_result
        print(f"     OpenRouter also failed.")

    if allow_fallback is not None:
        print(f"     Giving up — keeping original text for this chunk.")
        api_call_stats["failed_chunks"] += 1
        return allow_fallback

    raise RuntimeError(f"Gemini (all keys) and OpenRouter fallback failed after {max_retries} retries.")


def extract_glossary_from_pdf(pdf_text: str) -> list[str]:
    """Extract a focused list of technical terms from the PDF slides."""
    if not pdf_text:
        return []

    glossary_prompt = """You are a computer science terminology extractor.

Given the text of lecture slides, extract a focused list of technical terms
that a student should know from this lecture.

Rules:
- Include: algorithms, data structures, CS concepts, mathematical terms, named methods, tools, protocols
- Include terms in both English and Arabic if both appear
- Exclude: common English words, professor names, university names, page numbers
- Output ONLY a plain list, one term per line, no numbering, no explanation
- Maximum 60 terms"""

    print("    Extracting lecture glossary from PDF ...")
    raw = gemini_request(glossary_prompt, pdf_text, allow_fallback="")
    terms = [t.strip() for t in raw.splitlines() if t.strip()]
    print(f"    Glossary extracted: {len(terms)} terms")
    return terms


#Step B: PDF-guided transcript correction
def correct_transcript_with_pdf(
    transcript: str,
    pdf_text: str,
    glossary: list[str],
    correction_chunk_words: int = 800,
) -> str:
    """
    Correct Whisper transcript using lecture PDF + extracted glossary.
    Any chunk that fails all retries (Gemini + OpenRouter) keeps its
    original (uncorrected) text rather than crashing the whole pipeline.
    """
    glossary_str = "\n".join(glossary) if glossary else "(none)"
    global_terms = ", ".join(TECHNICAL_TERMS[:50])
    pdf_excerpt  = pdf_text[:4000]

    correction_system = f"""You are a transcript corrector for Arabic speech recognition output.

CRITICAL RULES — violating any of these is a failure:
1. Output length MUST be similar to input length. Do NOT summarize, shorten, or omit.
2. Preserve ALL Arabic sentences exactly. Fix only mis-heard technical terms.
3. Do NOT translate anything into English.
4. Do NOT reorder, restructure, or paraphrase.
5. If unsure whether something is an error, leave it unchanged.
6. Output ONLY the corrected Arabic transcript — nothing else.

What you MAY fix:
- A garbled word that phonetically matches a term from the glossary or slides.
- Clear Whisper hallucinations (repeated phrases, nonsense characters).

Lecture-specific terms (from slides):
{glossary_str}

General CS terms for reference:
{global_terms}

Lecture slides excerpt (for terminology reference):
{pdf_excerpt}"""

    chunks = chunk_text(transcript, correction_chunk_words)
    print(f"    Correcting {len(chunks)} chunk(s) with PDF + glossary ...")
    corrected_chunks = []

    for i, chunk in enumerate(chunks, 1):
        print(f"    Chunk {i}/{len(chunks)} ({len(chunk.split())} words) ...")
        corrected_chunk = gemini_request(
            correction_system, chunk, allow_fallback=chunk
        )

        ratio = len(corrected_chunk) / max(len(chunk), 1)
        if ratio < 0.4:
            print(f"        Chunk {i} shrank to {ratio:.0%} — using original chunk instead.")
            corrected_chunks.append(chunk)
        else:
            corrected_chunks.append(corrected_chunk)

    corrected = "\n\n".join(corrected_chunks)
    overall_ratio = len(corrected) / max(len(transcript), 1)
    print(f"    Correction done. ({len(transcript)} → {len(corrected)} chars, {overall_ratio:.0%})")
    return corrected


# Step C: Arabic -> English translation
def build_translation_prompt(glossary: list[str]) -> str:
    """Build a strong translation system prompt with concrete examples."""
    example_terms = glossary[:10] if glossary else [
        "Transformer", "Gradient Descent", "Kalman Filter",
        "Convex Hull", "Dijkstra", "RSA", "AES",
    ]
    examples_str  = "\n".join(f"  {t}" for t in example_terms)
    all_terms_str = ", ".join(TECHNICAL_TERMS)

    return f"""You are a professional Arabic-to-English translator specializing in computer science lectures.

Rules:
- Translate the Arabic text to fluent, natural English.
- If a technical term appears in English within the Arabic text, keep it EXACTLY as written.
- Do not translate, paraphrase, or modify any technical term.
- Keep proper nouns, names, and abbreviations unchanged.
- Maintain paragraph structure from the original.
- Output ONLY the translated English text. No notes, no commentary.

Examples of terms that must be preserved exactly as-is:
{examples_str}

Full list of known CS terms to preserve:
{all_terms_str}"""


def translate_with_gemini(text: str, glossary: list[str]) -> str:
    """Translate Arabic -> English in chunks using Gemini (falls back to OpenRouter per-chunk)."""
    system_prompt = build_translation_prompt(glossary)
    chunks = chunk_text(text, CHUNK_SIZE_WORDS)
    print(f"    Translating {len(chunks)} chunk(s) ...")
    translated = []

    for i, chunk in enumerate(chunks, 1):
        print(f"    Chunk {i}/{len(chunks)} ({len(chunk.split())} words) ...")
        result = gemini_request(system_prompt, chunk, allow_fallback=chunk)
        translated.append(result)

    return "\n\n".join(translated)


def translate_with_nllb(text: str) -> str:
    """Offline fallback: facebook/nllb-200-distilled-600M."""
    from transformers import pipeline as hf_pipeline
    import torch
    print("    Loading NLLB-200 (~1.2 GB, first run only) ...")
    translator = hf_pipeline(
        "translation",
        model="facebook/nllb-200-distilled-600M",
        src_lang="arb_Arab", tgt_lang="eng_Latn",
        device=0 if torch.cuda.is_available() else -1,
        max_length=512,
    )
    print("    NLLB-200 loaded.")
    chunks = chunk_text(text, max_words=150)
    translated = []
    for i, chunk in enumerate(chunks, 1):
        print(f"    Chunk {i}/{len(chunks)} ...")
        translated.append(translator(chunk)[0]["translation_text"].strip())
    return "\n\n".join(translated)

In [16]:
english_translation = ""
corrected_arabic    = ""
lecture_glossary    = []

clean_arabic_text = output["clean_text"]

if not clean_arabic_text.strip():
    print("  No Arabic text found — run the transcription cells first.")

elif TRANSLATION_MODE == "gemini":
    if not any(GEMINI_API_KEYS):
        print("  GEMINI_API_KEYS is empty.")
    else:
        if PDF_TEXT and USE_PDF_CORRECTION:
            print(" Step A — Extracting lecture glossary from PDF ...")
            lecture_glossary = extract_glossary_from_pdf(PDF_TEXT)
            if lecture_glossary:
                print("\nGlossary preview (first 15 terms):")
                for t in lecture_glossary[:15]:
                    print(f"   • {t}")
            time.sleep(5)

            print("\n Step B — Correcting transcript with PDF + glossary ...")
            
            corrected_arabic = correct_transcript_with_pdf(
                clean_arabic_text, PDF_TEXT, lecture_glossary
            )
            time.sleep(5)
        else:
            print("  No PDF — skipping glossary extraction and transcript correction.")
            corrected_arabic = clean_arabic_text

        print("\n Step C — Translating Arabic → English ...")
        english_translation = translate_with_gemini(
            corrected_arabic, lecture_glossary
        )

        print("\n Pipeline complete!")
        print(f"   Arabic (original) : {len(clean_arabic_text.split())} words")
        if corrected_arabic != clean_arabic_text:
            print(f"   Arabic (corrected): {len(corrected_arabic.split())} words")
        print(f"   English           : {len(english_translation.split())} words")
        if lecture_glossary:
            print(f"   Glossary terms    : {len(lecture_glossary)}")
            
        print("\n API Usage Stats:")
        for k, v in api_call_stats.items():
            print(f"   - {k}: {v}")

elif TRANSLATION_MODE == "local":
    print(" Translating Arabic → English with NLLB-200 (local) ...")
    corrected_arabic    = clean_arabic_text
    english_translation = translate_with_nllb(clean_arabic_text)
    print(f"\n Done. English: {len(english_translation.split())} words")

else:
    print(f" Unknown TRANSLATION_MODE: '{TRANSLATION_MODE}'. Use 'gemini' or 'local'.")

if english_translation:
    print()
    print("=" * 65)
    print("🇬🇧 ENGLISH TRANSLATION (first 800 chars)")
    print("=" * 65)
    print(english_translation[:800], "..." if len(english_translation) > 800 else "")

 Step A — Extracting lecture glossary from PDF ...
    Extracting lecture glossary from PDF ...
    Glossary extracted: 36 terms

Glossary preview (first 15 terms):
   • Virtual Memory
   • Operating Systems
   • Physical memory
   • Virtual memory
   • multiprogramming
   • Address translation
   • MMU
   • RAM
   • DISK
   • Virtual/Logical
   • Process
   • Segmentation
   • Paging
   • Fault Handler
   • Locality

 Step B — Correcting transcript with PDF + glossary ...
    Correcting 9 chunk(s) with PDF + glossary ...
    Chunk 1/9 (671 words) ...
    Chunk 2/9 (188 words) ...
    Chunk 3/9 (800 words) ...
    Chunk 4/9 (351 words) ...
    Chunk 5/9 (609 words) ...
     HTTP 503 — retrying in 10s (attempt 1/5) ...
     HTTP 503 — retrying in 20s (attempt 2/5) ...
    Chunk 6/9 (800 words) ...
    Chunk 7/9 (722 words) ...
    Chunk 8/9 (647 words) ...
     HTTP 503 — retrying in 10s (attempt 1/5) ...
     HTTP 503 — retrying in 20s (attempt 2/5) ...
    Chunk 9/9 (764 words) ...
  

## Save Outputs

In [19]:
import json

OUTPUT_DIR = "/kaggle/working" if IS_KAGGLE else "/tmp"

json_path = os.path.join(OUTPUT_DIR, "arabic_transcription_full.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({
        "source_file":           os.path.basename(AUDIO_PATH),
        "whisper_model":         WHISPER_MODEL_SIZE,
        "detected_language":     detected_lang,
        "total_segments":        output["total_segments"],
        "total_paragraphs":      output["total_paragraphs"],
        "raw_arabic_text":       output["clean_text"],
        "corrected_arabic_text": corrected_arabic if corrected_arabic else output["clean_text"],
        "lecture_glossary":      lecture_glossary,
        "english_translation":   english_translation,
        "timestamped_transcript":output["timestamped_transcript"],
    }, f, indent=2, ensure_ascii=False)

txt_path = os.path.join(OUTPUT_DIR, "arabic_clean_transcript.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(corrected_arabic if corrected_arabic else output["clean_text"])

print(" Saved:")
print(f"   \U0001F4C4 Full JSON         \u2192 {json_path}")
print(f"   \U0001F4C4 Arabic transcript \u2192 {txt_path}")

if english_translation:
    en_path = os.path.join(OUTPUT_DIR, "english_translation.txt")
    with open(en_path, "w", encoding="utf-8") as f:
        f.write(english_translation)
    print(f"   \U0001F4C4 English          \u2192 {en_path}")

if lecture_glossary:
    gl_path = os.path.join(OUTPUT_DIR, "lecture_glossary.txt")
    with open(gl_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lecture_glossary))
    print(f"   \U0001F4C4 Glossary         \u2192 {gl_path}")


 Saved:
   📄 Full JSON         → /kaggle/working/arabic_transcription_full.json
   📄 Arabic transcript → /kaggle/working/arabic_clean_transcript.txt
   📄 English          → /kaggle/working/english_translation.txt
   📄 Glossary         → /kaggle/working/lecture_glossary.txt
